In [11]:
from pixasonics.core import App, Mapper
from pixasonics.features import MeanChannelValue
from pixasonics.synths import Theremin

# create a new app
app = App()

# load an image from file
app.load_image_file("ecco2_and_grid.jpg")

app.audio = True
app.master_volume = 7 # check the slider after evaluating this cell

# create a Feature that will report the mean value of the red channel
mean_red = MeanChannelValue(filter_channels=0, name="MeanRed")
# attach the feature to the app
app.attach(mean_red)

# create a Theremin synth
theremin = Theremin(name="MySine")
# attach the Theremin to the app
app.attach(theremin)

# create a Mapper that will map the mean red pixel value to Theremin frequency
red2freq = Mapper(mean_red, theremin["frequency"], exponent=2, name="Red2Freq")
# attach the Mapper to the app
app.attach(red2freq)

Box(children=(Box(layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_right='…

In [12]:
# %pip install opencv-python

In [13]:
# import cv2
# import numpy as np
# import wave
# import subprocess
# import math

# from pixasonics.core import App, Mapper
# from pixasonics.features import MeanChannelValue
# from pixasonics.synths import Theremin

# VIDEO_PATH = "input.mp4"       # Kendi video dosyanın yolu
# AUDIO_PATH = "sonification.wav"
# OUTPUT_PATH = "sonification.mp4"

# PATTERN = "horizontal"         # "horizontal" veya "vertical"
# SONIFY_FPS = 4                 # Her saniyede kaç video karesi sonifiye edilsin
# IMAGE_SIZE = 500               # Pixasonics'e verilecek kare boyutu

# # Videoyu aç ve temel bilgilerini al
# cap = cv2.VideoCapture(VIDEO_PATH)
# if not cap.isOpened():
#     raise FileNotFoundError(f"Video açılamadı: {VIDEO_PATH}")

# video_fps = cap.get(cv2.CAP_PROP_FPS)
# frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# if video_fps <= 0 or frame_count <= 0:
#     cap.release()
#     raise ValueError("Videonun FPS veya kare sayısı okunamadı.")

# video_duration = frame_count / video_fps
# segment_duration = 1.0 / SONIFY_FPS

# # Pixasonics uygulaması ve ses eşlemesi
# app = App(
#     image_size=(IMAGE_SIZE, IMAGE_SIZE),
#     headless=True,
#     nrt=True,
# )

# mean_red = MeanChannelValue(filter_channels=0, name="MeanRed")
# theremin = Theremin(name="VideoSonification")
# red_to_frequency = Mapper(
#     mean_red,
#     theremin["frequency"],
#     exponent=2,
#     name="RedToFrequency",
# )

# app.attach(theremin)
# app.attach(red_to_frequency)

# audio_segments = []
# num_segments = math.ceil(video_duration * SONIFY_FPS)

# for i in range(num_segments):
#     time_position = i * segment_duration
#     this_duration = min(segment_duration, video_duration - time_position)

#     # Videodan bu zamana en yakın kareyi al
#     cap.set(cv2.CAP_PROP_POS_MSEC, time_position * 1000)
#     ok, frame_bgr = cap.read()
#     if not ok:
#         break

#     # OpenCV BGR okur; Pixasonics'e RGB veriyoruz
#     frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
#     frame_rgb = cv2.resize(frame_rgb, (IMAGE_SIZE, IMAGE_SIZE))

#     app.load_image_data(frame_rgb)

#     # Feature'ı görüntü yüklendikten sonra bağla
#     if i == 0:
#         app.attach(mean_red)

#     if PATTERN == "horizontal":
#         start = {
#             "probe_width": 1,
#             "probe_height": IMAGE_SIZE,
#             "probe_x": 0,
#             "probe_y": 0,
#         }
#         end = {"probe_x": IMAGE_SIZE - 1}
#     elif PATTERN == "vertical":
#         start = {
#             "probe_width": IMAGE_SIZE,
#             "probe_height": 1,
#             "probe_x": 0,
#             "probe_y": 0,
#         }
#         end = {"probe_y": IMAGE_SIZE - 1}
#     else:
#         cap.release()
#         raise ValueError("PATTERN 'horizontal' veya 'vertical' olmalı.")

#     timeline = [
#         (0, start),
#         (this_duration, end),
#     ]

#     # Bu kare için otomatik ses parçası üret
#     audio_segments.append(app.render_timeline_to_array(timeline))
#     print(f"{i + 1}/{num_segments} kare işlendi")

# cap.release()

# if not audio_segments:
#     raise RuntimeError("Videodan işlenecek kare alınamadı.")

# # Ses parçalarını birleştir (Pixasonics dizisi: kanal x örnek)
# audio = np.concatenate(audio_segments, axis=1)
# sample_rate = int(app.graph.sample_rate)

# # WAV dosyasına stereo 16-bit PCM olarak kaydet
# peak = np.max(np.abs(audio))
# if peak > 0:
#     audio = audio / peak * 0.95

# pcm = (np.clip(audio.T, -1, 1) * 32767).astype("<i2")

# with wave.open(AUDIO_PATH, "wb") as wav_file:
#     wav_file.setnchannels(2)
#     wav_file.setsampwidth(2)
#     wav_file.setframerate(sample_rate)
#     wav_file.writeframes(pcm.tobytes())

# # Üretilen sesi orijinal video görüntüsüne ekle
# subprocess.run(
#     [
#         "ffmpeg", "-y",
#         "-i", VIDEO_PATH,
#         "-i", AUDIO_PATH,
#         "-map", "0:v:0",
#         "-map", "1:a:0",
#         "-c:v", "libx264",
#         "-pix_fmt", "yuv420p",
#         "-c:a", "aac",
#         "-shortest",
#         OUTPUT_PATH,
#     ],
#     check=True,
# )

# print("Video kaydedildi:", OUTPUT_PATH)

In [14]:
# import shutil
# print(shutil.which("ffmpeg"))

In [15]:
# from pathlib import Path
# import re
# import subprocess
# import wave

# import cv2
# import numpy as np
# from pixasonics.core import App, Mapper
# from pixasonics.features import MeanChannelValue
# from pixasonics.synths import Oscillator


# FRAME_DIR = Path("./frames_0")  # Karelerin bulunduğu klasör
# AUDIO_PATH = "sonification.wav"
# TEMP_VIDEO_PATH = "frames_video.mp4"
# OUTPUT_PATH = "sonification.mp4"

# SONIFY_FPS = 4       # Karelerin videoda saniyedeki gösterim sayısı
# IMAGE_SIZE = 500


# def natural_sort_key(path):
#     # frame_2.png, frame_10.png şeklinde doğal sıralama yapar
#     return [
#         int(part) if part.isdigit() else part.lower()
#         for part in re.split(r"(\d+)", path.name)
#     ]


# frame_paths = sorted(
#     [
#         p for p in FRAME_DIR.iterdir()
#         if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
#     ],
#     key=natural_sort_key,
# )

# if not frame_paths:
#     raise FileNotFoundError(f"{FRAME_DIR} klasöründe resim karesi bulunamadı.")

# app = App(image_size=(IMAGE_SIZE, IMAGE_SIZE), headless=True, nrt=True)

# mean_red = MeanChannelValue(filter_channels=0, name="MeanRed")

# # Daha tok, yumuşak ses: alçak frekanslı üçgen dalga ve low-pass filtre
# synth = Oscillator(
#     frequency=100,
#     waveform="triangle",
#     lp_cutoff=700,
#     name="TemperatureBass",
# )

# mapper = Mapper(
#     mean_red,
#     synth["frequency"],
#     out_low=65,
#     out_high=180,
#     exponent=1,
#     name="TemperatureToBass",
# )

# app.attach(synth)
# app.attach(mapper)

# video_writer = cv2.VideoWriter(
#     TEMP_VIDEO_PATH,
#     cv2.VideoWriter_fourcc(*"mp4v"),
#     SONIFY_FPS,
#     (IMAGE_SIZE, IMAGE_SIZE),
# )
# if not video_writer.isOpened():
#     raise RuntimeError(f"Video dosyası oluşturulamadı: {TEMP_VIDEO_PATH}")

# audio_segments = []
# segment_duration = 1.0 / SONIFY_FPS

# try:
#     for i, frame_path in enumerate(frame_paths):
#         frame_bgr = cv2.imread(str(frame_path))
#         if frame_bgr is None:
#             raise ValueError(f"Kare okunamadı: {frame_path}")

#         frame_bgr = cv2.resize(frame_bgr, (IMAGE_SIZE, IMAGE_SIZE))
#         frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

#         # Bu video karesi Pixasonics'e yüklenir
#         app.load_image_data(frame_rgb)

#         if i == 0:
#             app.attach(mean_red)

#         # Çizgi/Probe: ilk karede üstte, ortada altta, son karede tekrar üstte
#         if len(frame_paths) > 1:
#             phase = i / (len(frame_paths) - 1)
#         else:
#             phase = 0

#         y = round((IMAGE_SIZE - 1) * (1 - abs(2 * phase - 1)))

#         # Ses, görüntünün çizginin geçtiği yatay satırından üretilir
#         probe = {
#             "probe_width": IMAGE_SIZE,
#             "probe_height": 1,
#             "probe_x": 0,
#             "probe_y": y,
#         }
#         timeline = [
#             (0, probe),
#             (segment_duration, probe),
#         ]
#         audio_segments.append(app.render_timeline_to_array(timeline))

#         # Videoda da aynı y konumuna yatay kırmızı çizgi çizilir
#         display_frame = frame_bgr.copy()
#         cv2.line(
#             display_frame,
#             (0, y),
#             (IMAGE_SIZE - 1, y),
#             (0, 0, 255),
#             2,
#         )
#         video_writer.write(display_frame)

#         print(f"{i + 1}/{len(frame_paths)} kare işlendi: {frame_path.name}")
# finally:
#     video_writer.release()

# if not audio_segments:
#     raise RuntimeError("Ses üretmek için hiçbir kare işlenemedi.")

# # Ses parçalarını birleştir
# audio = np.concatenate(audio_segments, axis=1)
# sample_rate = int(app.graph.sample_rate)

# peak = np.max(np.abs(audio))
# if peak > 0:
#     audio = audio / peak * 0.95

# pcm = (np.clip(audio.T, -1, 1) * 32767).astype("<i2")

# with wave.open(AUDIO_PATH, "wb") as wav_file:
#     wav_file.setnchannels(2)
#     wav_file.setsampwidth(2)
#     wav_file.setframerate(sample_rate)
#     wav_file.writeframes(pcm.tobytes())

# # Görüntü videosuyla Pixasonics sesini birleştir
# subprocess.run(
#     [
#         "ffmpeg", "-y",
#         "-i", TEMP_VIDEO_PATH,
#         "-i", AUDIO_PATH,
#         "-map", "0:v:0",
#         "-map", "1:a:0",
#         "-c:v", "libx264",
#         "-pix_fmt", "yuv420p",
#         "-c:a", "aac",
#         "-shortest",
#         OUTPUT_PATH,
#     ],
#     check=True,
# )

# print("Birleşik video kaydedildi:", OUTPUT_PATH)

In [16]:
# from pathlib import Path
# import re
# import shutil
# import subprocess
# import wave

# import cv2
# import numpy as np
# from pixasonics.core import App, Mapper
# from pixasonics.features import MeanChannelValue
# from pixasonics.synths import Oscillator


# # --- Ayarlar ---
# FRAME_DIR = Path("./frames_0")
# AUDIO_PATH = "sonification.wav"
# TEMP_VIDEO_PATH = "frames_video.mp4"
# OUTPUT_PATH = "sonification.mp4"

# VIDEO_FPS = 30
# SECONDS_PER_IMAGE = 1.0
# IMAGE_SIZE = 500


# def natural_sort_key(path):
#     # frame_2.png, frame_10.png şeklinde doğal sıralar
#     return [
#         int(part) if part.isdigit() else part.lower()
#         for part in re.split(r"(\d+)", path.name)
#     ]


# frame_paths = sorted(
#     [
#         p for p in FRAME_DIR.iterdir()
#         if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
#     ],
#     key=natural_sort_key,
# )

# if not frame_paths:
#     raise FileNotFoundError(f"{FRAME_DIR} klasöründe resim karesi bulunamadı.")

# frames_per_image = round(SECONDS_PER_IMAGE * VIDEO_FPS)
# if frames_per_image < 1:
#     raise ValueError("SECONDS_PER_IMAGE * VIDEO_FPS en az 1 olmalı.")

# # Kare sayısına tam bölünebilir süre kullan
# segment_duration = frames_per_image / VIDEO_FPS
# total_segments = len(frame_paths)


# def scan_y(progress):
#     """0..1 ilerlemede çizgiyi üstten alta ve tekrar üste taşır."""
#     progress = max(0.0, min(1.0, progress))
#     return round((IMAGE_SIZE - 1) * (1 - abs(2 * progress - 1)))


# def probe_settings(y):
#     return {
#         "probe_width": IMAGE_SIZE,
#         "probe_height": 1,
#         "probe_x": 0,
#         "probe_y": y,
#     }


# # --- Pixasonics ses zinciri ---
# app = App(image_size=(IMAGE_SIZE, IMAGE_SIZE), headless=True, nrt=True)

# mean_red = MeanChannelValue(filter_channels=0, name="MeanRed")

# # Alçak, yumuşak üçgen dalga ve low-pass filtre
# synth = Oscillator(
#     frequency=100,
#     waveform="triangle",
#     lp_cutoff=700,
#     name="TemperatureBass",
# )

# mapper = Mapper(
#     mean_red,
#     synth["frequency"],
#     out_low=65,
#     out_high=180,
#     exponent=1,
#     name="TemperatureToBass",
# )

# app.attach(synth)
# app.attach(mapper)


# # --- Video ve ses üretimi ---
# video_writer = cv2.VideoWriter(
#     TEMP_VIDEO_PATH,
#     cv2.VideoWriter_fourcc(*"mp4v"),
#     VIDEO_FPS,
#     (IMAGE_SIZE, IMAGE_SIZE),
# )
# if not video_writer.isOpened():
#     raise RuntimeError(f"Video dosyası oluşturulamadı: {TEMP_VIDEO_PATH}")

# audio_segments = []

# try:
#     for i, frame_path in enumerate(frame_paths):
#         frame_bgr = cv2.imread(str(frame_path))
#         if frame_bgr is None:
#             raise ValueError(f"Kare okunamadı: {frame_path}")

#         frame_bgr = cv2.resize(frame_bgr, (IMAGE_SIZE, IMAGE_SIZE))
#         frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

#         # Bu kareyi Pixasonics'e yükle
#         app.load_image_data(frame_rgb)
#         if i == 0:
#             app.attach(mean_red)

#         # Bu karenin video boyunca kapladığı hareket aralığı
#         start_progress = i / total_segments
#         end_progress = (i + 1) / total_segments

#         # Probe çizgisini aynı kare üzerinde hareket ettirerek sesi üret
#         timeline = [
#             (0, probe_settings(scan_y(start_progress))),
#         ]

#         # Üst-alt dönüş noktası bu karenin içindeyse ara nokta ekle
#         if start_progress < 0.5 < end_progress:
#             turn_time = (
#                 (0.5 - start_progress)
#                 / (end_progress - start_progress)
#                 * segment_duration
#             )
#             timeline.append(
#                 (turn_time, probe_settings(IMAGE_SIZE - 1))
#             )

#         timeline.append(
#             (segment_duration, probe_settings(scan_y(end_progress)))
#         )

#         audio_segments.append(app.render_timeline_to_array(timeline))

#         # Kırmızı çizgiyi videoda aynı hareketle göster
#         for frame_index in range(frames_per_image):
#             progress = (
#                 i + frame_index / frames_per_image
#             ) / total_segments
#             y = scan_y(progress)

#             display_frame = frame_bgr.copy()
#             cv2.line(
#                 display_frame,
#                 (0, y),
#                 (IMAGE_SIZE - 1, y),
#                 (0, 0, 255),
#                 2,
#             )
#             video_writer.write(display_frame)

#         print(f"{i + 1}/{total_segments} kare işlendi: {frame_path.name}")

# finally:
#     video_writer.release()


# if not audio_segments:
#     raise RuntimeError("Ses üretmek için hiçbir kare işlenemedi.")


# # --- WAV sesini kaydet ---
# audio = np.concatenate(audio_segments, axis=1)
# sample_rate = int(app.graph.sample_rate)

# peak = np.max(np.abs(audio))
# if peak > 0:
#     audio = audio / peak * 0.95

# pcm = (np.clip(audio.T, -1, 1) * 32767).astype("<i2")

# with wave.open(AUDIO_PATH, "wb") as wav_file:
#     wav_file.setnchannels(2)
#     wav_file.setsampwidth(2)
#     wav_file.setframerate(sample_rate)
#     wav_file.writeframes(pcm.tobytes())


# # --- Video ve sesi birleştir ---
# ffmpeg_path = shutil.which("ffmpeg")
# if ffmpeg_path is None:
#     raise FileNotFoundError(
#         "ffmpeg bulunamadı. Terminal'de 'brew install ffmpeg' çalıştırıp "
#         "notebook kernel'ini yeniden başlat."
#     )

# subprocess.run(
#     [
#         ffmpeg_path, "-y",
#         "-i", TEMP_VIDEO_PATH,
#         "-i", AUDIO_PATH,
#         "-map", "0:v:0",
#         "-map", "1:a:0",
#         "-c:v", "libx264",
#         "-pix_fmt", "yuv420p",
#         "-c:a", "aac",
#         "-shortest",
#         OUTPUT_PATH,
#     ],
#     check=True,
# )

# print(f"Birleşik video kaydedildi: {OUTPUT_PATH}")
# print(f"Süre yaklaşık {total_segments * segment_duration:.1f} saniye.")

In [17]:
# from pathlib import Path
# import re
# import shutil
# import subprocess
# import wave

# import cv2
# import numpy as np
# from pixasonics.core import App, Mapper
# from pixasonics.features import MeanChannelValue
# from pixasonics.synths import Oscillator


# # --- Ayarlar ---
# FRAME_DIR = Path("./frames_0")
# AUDIO_PATH = "sonification.wav"
# TEMP_VIDEO_PATH = "frames_video.mp4"
# OUTPUT_PATH = "sonification.mp4"

# VIDEO_FPS = 30
# SECONDS_PER_IMAGE = 1.0
# IMAGE_SIZE = 500


# def natural_sort_key(path):
#     # frame_2.png, frame_10.png şeklinde doğal sıralar
#     return [
#         int(part) if part.isdigit() else part.lower()
#         for part in re.split(r"(\d+)", path.name)
#     ]


# def scan_y(progress):
#     """Çizgiyi tüm video boyunca üstten alta, sonra tekrar üste taşır."""
#     progress = max(0.0, min(1.0, progress))
#     return round((IMAGE_SIZE - 1) * (1 - abs(2 * progress - 1)))


# def probe_settings(y):
#     return {
#         "probe_width": IMAGE_SIZE,
#         "probe_height": 1,
#         "probe_x": 0,
#         "probe_y": y,
#     }


# def make_color_features(frame_rgb):
#     """
#     RGB renklerini Pixasonics için üç anlamsal kanala dönüştürür:
#     0: kırmızıdan sarıya giden ton değeri
#     1: aynı ton değeri, filtre kesimini kontrol eder
#     2: kırmızılık, ses seviyesini kontrol eder
#     """
#     rgb = frame_rgb.astype(np.float32) / 255.0
#     red = rgb[:, :, 0]
#     green = rgb[:, :, 1]
#     blue = rgb[:, :, 2]

#     max_rgb = np.max(rgb, axis=2)
#     min_rgb = np.min(rgb, axis=2)
#     saturation = (max_rgb - min_rgb) / (max_rgb + 1e-6)

#     # Kırmızı: R, G ve B'den belirgin biçimde büyük
#     red_score = np.clip(
#         (red - np.maximum(green, blue)) / (red + 1e-6),
#         0,
#         1,
#     ) * saturation

#     # Sarı: R ve G yüksek, B düşük
#     yellow_score = np.clip(
#         (np.minimum(red, green) - blue)
#         / (np.maximum(red, green) + 1e-6),
#         0,
#         1,
#     ) * saturation

#     # Kırmızı -> 0 (tok), sarı -> 1 (parlak).
#     # Kırmızı/sarı olmayan renkler nötr orta değere çekilir.
#     warm_confidence = np.maximum(red_score, yellow_score)
#     tone_score = yellow_score + (1.0 - warm_confidence) * 0.5

#     return np.stack(
#         (tone_score, tone_score, red_score),
#         axis=2,
#     ).astype(np.float32)


# frame_paths = sorted(
#     [
#         p for p in FRAME_DIR.iterdir()
#         if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
#     ],
#     key=natural_sort_key,
# )

# if not frame_paths:
#     raise FileNotFoundError(f"{FRAME_DIR} klasöründe resim karesi bulunamadı.")

# frames_per_image = round(SECONDS_PER_IMAGE * VIDEO_FPS)
# if frames_per_image < 1:
#     raise ValueError("SECONDS_PER_IMAGE * VIDEO_FPS en az 1 olmalı.")

# segment_duration = frames_per_image / VIDEO_FPS
# total_segments = len(frame_paths)


# # --- Pixasonics ve synth ---
# app = App(image_size=(IMAGE_SIZE, IMAGE_SIZE), headless=True, nrt=True)

# # Kanal 0 sarı-kırmızı tonunu; kanal 1 parlaklık filtresini;
# # kanal 2 kırmızı sıcak bölgeleri temsil eder.
# mean_tone = MeanChannelValue(filter_channels=0, name="WarmColorTone")
# mean_brightness = MeanChannelValue(filter_channels=1, name="ColorBrightness")
# mean_hot_red = MeanChannelValue(filter_channels=2, name="HotRed")

# # Synth değişmiyor: üçgen dalga, sıcak bölgelerde alçak/tok ses.
# synth = Oscillator(
#     frequency=100,
#     waveform="triangle",
#     lp_cutoff=700,
#     name="SurfaceTemperature",
# )

# # Kırmızıdan sarıya: alçak frekanstan daha yüksek frekansa
# tone_to_frequency = Mapper(
#     mean_tone,
#     synth["frequency"],
#     in_low=0,
#     in_high=1,
#     out_low=70,
#     out_high=260,
#     name="RedToYellowPitch",
# )

# # Sarı bölgeler daha parlak; kırmızı bölgeler daha koyu/tok
# tone_to_filter = Mapper(
#     mean_brightness,
#     synth["lp_cutoff"],
#     in_low=0,
#     in_high=1,
#     out_low=250,
#     out_high=3500,
#     name="ColorToBrightness",
# )

# # Kırmızı alanlar daha güçlü duyulsun
# red_to_amplitude = Mapper(
#     mean_hot_red,
#     synth["amplitude"],
#     in_low=0,
#     in_high=1,
#     out_low=0.25,
#     out_high=0.8,
#     name="HotRedToAmplitude",
# )

# app.attach(synth)
# app.attach(tone_to_frequency)
# app.attach(tone_to_filter)
# app.attach(red_to_amplitude)


# # --- Video ve ses üretimi ---
# video_writer = cv2.VideoWriter(
#     TEMP_VIDEO_PATH,
#     cv2.VideoWriter_fourcc(*"mp4v"),
#     VIDEO_FPS,
#     (IMAGE_SIZE, IMAGE_SIZE),
# )
# if not video_writer.isOpened():
#     raise RuntimeError(f"Video dosyası oluşturulamadı: {TEMP_VIDEO_PATH}")

# audio_segments = []

# try:
#     for i, frame_path in enumerate(frame_paths):
#         frame_bgr = cv2.imread(str(frame_path))
#         if frame_bgr is None:
#             raise ValueError(f"Kare okunamadı: {frame_path}")

#         frame_bgr = cv2.resize(frame_bgr, (IMAGE_SIZE, IMAGE_SIZE))
#         frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

#         # Pixasonics'e RGB yerine renklerden üretilen 3 özellik kanalını ver.
#         feature_image = make_color_features(frame_rgb)
#         app.load_image_data(feature_image)

#         if i == 0:
#             app.attach(mean_tone)
#             app.attach(mean_brightness)
#             app.attach(mean_hot_red)

#         start_progress = i / total_segments
#         end_progress = (i + 1) / total_segments

#         timeline = [
#             (0, probe_settings(scan_y(start_progress))),
#         ]

#         # Tarama dönüş noktasını doğru zamanda ekle
#         if start_progress < 0.5 < end_progress:
#             turn_time = (
#                 (0.5 - start_progress)
#                 / (end_progress - start_progress)
#                 * segment_duration
#             )
#             timeline.append(
#                 (turn_time, probe_settings(IMAGE_SIZE - 1))
#             )

#         timeline.append(
#             (segment_duration, probe_settings(scan_y(end_progress)))
#         )

#         audio_segments.append(app.render_timeline_to_array(timeline))

#         # Görüntüde, sesin taradığı satırla aynı çizgiyi göster
#         for frame_index in range(frames_per_image):
#             progress = (
#                 i + frame_index / frames_per_image
#             ) / total_segments
#             y = scan_y(progress)

#             display_frame = frame_bgr.copy()
#             cv2.line(
#                 display_frame,
#                 (0, y),
#                 (IMAGE_SIZE - 1, y),
#                 (0, 0, 255),
#                 2,
#             )
#             video_writer.write(display_frame)

#         print(f"{i + 1}/{total_segments} kare işlendi: {frame_path.name}")

# finally:
#     video_writer.release()


# if not audio_segments:
#     raise RuntimeError("Ses üretmek için hiçbir kare işlenemedi.")


# # --- WAV sesini kaydet ---
# audio = np.concatenate(audio_segments, axis=1)
# sample_rate = int(app.graph.sample_rate)

# peak = np.max(np.abs(audio))
# if peak > 0:
#     audio = audio / peak * 0.95

# pcm = (np.clip(audio.T, -1, 1) * 32767).astype("<i2")

# with wave.open(AUDIO_PATH, "wb") as wav_file:
#     wav_file.setnchannels(2)
#     wav_file.setsampwidth(2)
#     wav_file.setframerate(sample_rate)
#     wav_file.writeframes(pcm.tobytes())


# # --- Video ve sesi birleştir ---
# ffmpeg_path = shutil.which("ffmpeg")
# if ffmpeg_path is None:
#     raise FileNotFoundError(
#         "ffmpeg bulunamadı. Terminal'de 'brew install ffmpeg' çalıştırıp "
#         "notebook kernel'ini yeniden başlat."
#     )

# subprocess.run(
#     [
#         ffmpeg_path, "-y",
#         "-i", TEMP_VIDEO_PATH,
#         "-i", AUDIO_PATH,
#         "-map", "0:v:0",
#         "-map", "1:a:0",
#         "-c:v", "libx264",
#         "-pix_fmt", "yuv420p",
#         "-c:a", "aac",
#         "-shortest",
#         OUTPUT_PATH,
#     ],
#     check=True,
# )

# print(f"Birleşik video kaydedildi: {OUTPUT_PATH}")
# print(f"Süre yaklaşık {total_segments * segment_duration:.1f} saniye.")

In [18]:
# from pathlib import Path
# import re
# import shutil
# import subprocess
# import wave

# import cv2
# import numpy as np
# from pixasonics.core import App, Mapper
# from pixasonics.features import MeanChannelValue
# from pixasonics.synths import SimpleFM


# # --- Ayarlar ---
# FRAME_DIR = Path("./frames_0")
# AUDIO_PATH = "sonification.wav"
# TEMP_VIDEO_PATH = "frames_video.mp4"
# OUTPUT_PATH = "sonification.mp4"

# VIDEO_FPS = 30
# SECONDS_PER_IMAGE = 1.0
# IMAGE_SIZE = 500


# def natural_sort_key(path):
#     """frame_2.png, frame_10.png şeklinde doğal sıralar."""
#     return [
#         int(part) if part.isdigit() else part.lower()
#         for part in re.split(r"(\d+)", path.name)
#     ]


# def scan_y(progress):
#     """Çizgiyi tüm video boyunca üstten alta, sonra tekrar üste taşır."""
#     progress = max(0.0, min(1.0, progress))
#     return round((IMAGE_SIZE - 1) * (1 - abs(2 * progress - 1)))


# def probe_settings(y):
#     return {
#         "probe_width": IMAGE_SIZE,
#         "probe_height": 1,
#         "probe_x": 0,
#         "probe_y": y,
#     }


# def make_color_features(frame_rgb):
#     """
#     RGB görüntüyü ses eşlemesi için üç kanala dönüştürür:
#     kanal 0: kırmızıdan sarıya ton (kırmızı düşük, sarı yüksek)
#     kanal 1: filtre parlaklığı (kırmızı koyu, sarı açık)
#     kanal 2: kırmızılık (kırmızı alanlarda ses seviyesi/tını yoğunluğu)
#     """
#     rgb = frame_rgb.astype(np.float32) / 255.0
#     red = rgb[:, :, 0]
#     green = rgb[:, :, 1]
#     blue = rgb[:, :, 2]

#     max_rgb = np.max(rgb, axis=2)
#     min_rgb = np.min(rgb, axis=2)
#     saturation = (max_rgb - min_rgb) / (max_rgb + 1e-6)

#     red_score = np.clip(
#         (red - np.maximum(green, blue)) / (red + 1e-6),
#         0,
#         1,
#     ) * saturation

#     yellow_score = np.clip(
#         (np.minimum(red, green) - blue)
#         / (np.maximum(red, green) + 1e-6),
#         0,
#         1,
#     ) * saturation

#     warm_confidence = np.maximum(red_score, yellow_score)

#     # Kırmızı 0 (tok), sarı 1 (parlak); diğer renkler nötr orta değer.
#     tone_score = yellow_score + (1.0 - warm_confidence) * 0.5

#     return np.stack(
#         (tone_score, tone_score, red_score),
#         axis=2,
#     ).astype(np.float32)


# # --- Kare dosyalarını sırala ---
# if not FRAME_DIR.is_dir():
#     raise FileNotFoundError(f"Kare klasörü bulunamadı: {FRAME_DIR}")

# frame_paths = sorted(
#     [
#         p for p in FRAME_DIR.iterdir()
#         if p.suffix.lower() in {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
#     ],
#     key=natural_sort_key,
# )

# if not frame_paths:
#     raise FileNotFoundError(f"{FRAME_DIR} klasöründe resim karesi bulunamadı.")

# frames_per_image = round(SECONDS_PER_IMAGE * VIDEO_FPS)
# if frames_per_image < 1:
#     raise ValueError("SECONDS_PER_IMAGE * VIDEO_FPS en az 1 olmalı.")

# segment_duration = frames_per_image / VIDEO_FPS
# total_segments = len(frame_paths)


# # --- Pixasonics ve SimpleFM synth ---
# app = App(image_size=(IMAGE_SIZE, IMAGE_SIZE), headless=True, nrt=True)

# mean_tone = MeanChannelValue(filter_channels=0, name="WarmColorTone")
# mean_filter = MeanChannelValue(filter_channels=1, name="ColorFilter")
# mean_hot_red = MeanChannelValue(filter_channels=2, name="HotRed")

# # Synth değişti: FM sintezi, alçak taşıyıcı frekans ve filtrelenmiş çıkış.
# synth = SimpleFM(
#     carrier_frequency=110,
#     harmonicity_ratio=1,
#     modulation_index=1,
#     lp_cutoff=2000,
#     name="TemperatureFM",
# )

# # Kırmızı düşük/tok, sarı daha yüksek/parlak duyulur.
# tone_to_frequency = Mapper(
#     mean_tone,
#     synth["carrier_freq"],
#     in_low=0,
#     in_high=1,
#     out_low=70,
#     out_high=260,
#     name="RedToYellowPitch",
# )

# # Kırmızıda daha dolgun FM tınısı; sarıda daha sade tını.
# red_to_timbre = Mapper(
#     mean_hot_red,
#     synth["mod_index"],
#     in_low=0,
#     in_high=1,
#     out_low=0.4,
#     out_high=3.0,
#     name="HotRedToTimbre",
# )

# # Sarı daha parlak, kırmızı daha koyu/tok.
# color_to_filter = Mapper(
#     mean_filter,
#     synth["lp_cutoff"],
#     in_low=0,
#     in_high=1,
#     out_low=300,
#     out_high=3500,
#     name="ColorToBrightness",
# )

# # Kırmızı bölgeler biraz daha yüksek ses verir.
# red_to_amplitude = Mapper(
#     mean_hot_red,
#     synth["amplitude"],
#     in_low=0,
#     in_high=1,
#     out_low=0.25,
#     out_high=0.8,
#     name="HotRedToAmplitude",
# )

# app.attach(synth)
# app.attach(tone_to_frequency)
# app.attach(red_to_timbre)
# app.attach(color_to_filter)
# app.attach(red_to_amplitude)


# # --- Video ve ses üretimi ---
# video_writer = cv2.VideoWriter(
#     TEMP_VIDEO_PATH,
#     cv2.VideoWriter_fourcc(*"mp4v"),
#     VIDEO_FPS,
#     (IMAGE_SIZE, IMAGE_SIZE),
# )

# if not video_writer.isOpened():
#     raise RuntimeError(f"Video dosyası oluşturulamadı: {TEMP_VIDEO_PATH}")

# audio_segments = []

# try:
#     for i, frame_path in enumerate(frame_paths):
#         frame_bgr = cv2.imread(str(frame_path))
#         if frame_bgr is None:
#             raise ValueError(f"Kare okunamadı: {frame_path}")

#         frame_bgr = cv2.resize(frame_bgr, (IMAGE_SIZE, IMAGE_SIZE))
#         frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

#         # Renk özelliklerini Pixasonics'e yükle
#         feature_image = make_color_features(frame_rgb)
#         app.load_image_data(feature_image)

#         if i == 0:
#             app.attach(mean_tone)
#             app.attach(mean_filter)
#             app.attach(mean_hot_red)

#         start_progress = i / total_segments
#         end_progress = (i + 1) / total_segments

#         # Aynı hareketi ses için Pixasonics Probe'unda uygula.
#         timeline = [
#             (0, probe_settings(scan_y(start_progress))),
#         ]

#         # Aşağıdaki dönüş noktasını doğru zamanda ekle.
#         if start_progress < 0.5 < end_progress:
#             turn_time = (
#                 (0.5 - start_progress)
#                 / (end_progress - start_progress)
#                 * segment_duration
#             )
#             timeline.append(
#                 (turn_time, probe_settings(IMAGE_SIZE - 1))
#             )

#         timeline.append(
#             (segment_duration, probe_settings(scan_y(end_progress)))
#         )

#         audio_segments.append(app.render_timeline_to_array(timeline))

#         # Görüntüde de ses taramasıyla aynı konumda çizgi göster.
#         for frame_index in range(frames_per_image):
#             progress = (
#                 i + frame_index / frames_per_image
#             ) / total_segments
#             y = scan_y(progress)

#             display_frame = frame_bgr.copy()
#             cv2.line(
#                 display_frame,
#                 (0, y),
#                 (IMAGE_SIZE - 1, y),
#                 (0, 0, 255),
#                 2,
#             )
#             video_writer.write(display_frame)

#         print(f"{i + 1}/{total_segments} kare işlendi: {frame_path.name}")

# finally:
#     video_writer.release()


# if not audio_segments:
#     raise RuntimeError("Ses üretmek için hiçbir kare işlenemedi.")


# # --- WAV sesini kaydet ---
# audio = np.concatenate(audio_segments, axis=1)
# sample_rate = int(app.graph.sample_rate)

# peak = np.max(np.abs(audio))
# if peak > 0:
#     audio = audio / peak * 0.95

# pcm = (np.clip(audio.T, -1, 1) * 32767).astype("<i2")

# with wave.open(AUDIO_PATH, "wb") as wav_file:
#     wav_file.setnchannels(2)
#     wav_file.setsampwidth(2)
#     wav_file.setframerate(sample_rate)
#     wav_file.writeframes(pcm.tobytes())


# # --- Video ve sesi birleştir ---
# ffmpeg_path = shutil.which("ffmpeg")
# if ffmpeg_path is None:
#     raise FileNotFoundError(
#         "ffmpeg bulunamadı. Terminal'de 'brew install ffmpeg' çalıştırıp "
#         "notebook kernel'ini yeniden başlat."
#     )

# subprocess.run(
#     [
#         ffmpeg_path, "-y",
#         "-i", TEMP_VIDEO_PATH,
#         "-i", AUDIO_PATH,
#         "-map", "0:v:0",
#         "-map", "1:a:0",
#         "-c:v", "libx264",
#         "-pix_fmt", "yuv420p",
#         "-c:a", "aac",
#         "-shortest",
#         OUTPUT_PATH,
#     ],
#     check=True,
# )

# print(f"Birleşik video kaydedildi: {OUTPUT_PATH}")
# print(f"Süre yaklaşık {total_segments * segment_duration:.1f} saniye.")